# Figures for the README — drawn from `outputs/*.csv` only

Run the analysis notebooks first (see *How to run* in the README). This notebook reads result CSVs from `outputs/` and writes PNGs to `images/`.
No result number is typed in here; every number in a figure, label or caption comes from a CSV.

- `TRANSPARENT = False`: white background, saved to `images/`.
- `TRANSPARENT = True`: transparent background for the portfolio site, saved to `images/transparent/`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib import font_manager
from matplotlib.colors import to_rgb
from matplotlib.patches import Rectangle

from config import OUT_DIR, FIG_DIR

TRANSPARENT = False   # True: transparent PNGs for the site -> images/transparent/
DPI = 200
SAVE_DIR = FIG_DIR / "transparent" if TRANSPARENT else FIG_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# palette (CLAUDE.md)
INK, INK2, INK3 = "#23233A", "#4A4A66", "#7B7B96"
VIOLET, ROSE, BLUE, MINT = "#7B6FD8", "#D9536B", "#4C7FE0", "#2F9E88"
GRID = "#E6E6EF"

def tint(color, a):
    """Blend a colour toward white; a=1 keeps the colour, a=0 is white."""
    return tuple(1 - (1 - np.array(to_rgb(color))) * a)

_available = {f.name for f in font_manager.fontManager.ttflist}
FONT = "IBM Plex Sans" if "IBM Plex Sans" in _available else "DejaVu Sans"
plt.rcParams.update({
    "font.family": FONT, "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK3, "ytick.color": INK3,
    "axes.edgecolor": GRID, "axes.linewidth": 1.0,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.facecolor": "none" if TRANSPARENT else "white",
    "figure.facecolor": "none" if TRANSPARENT else "white",
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.unicode_minus": True,
})
print("font:", FONT, "| transparent:", TRANSPARENT, "| save dir:", SAVE_DIR.name)

def read(name):
    return pd.read_csv(OUT_DIR / name)

def head(fig, title, subtitle, top=0.80):
    fig.subplots_adjust(top=top)
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    x0 = min(ax.get_tightbbox(r).x0 for ax in fig.axes)          # left edge of axis labels
    x = fig.transFigure.inverted().transform((x0, 0))[0]
    fig.text(x, 0.995, title, ha="left", va="top", fontsize=14, fontweight="bold", color=INK)
    fig.text(x, 0.995 - 0.075, subtitle, ha="left", va="top", fontsize=9, color=INK3, linespacing=1.5)

def save(fig, name):
    fig.savefig(SAVE_DIR / f"{name}.png", dpi=DPI, bbox_inches="tight", pad_inches=0.25,
                transparent=TRANSPARENT, facecolor="none" if TRANSPARENT else "white")
    plt.close(fig)
    print("saved", SAVE_DIR.name + "/" + name + ".png")

def sgn(v, nd=2):
    """Signed number with a true minus sign."""
    return f"{v:+.{nd}f}".replace("-", "\u2212")

def neg(s):
    return s.replace("-", "\u2212")

LABELS = {
    "has_demo": "Demographic survey present", "sales_early": "Baseline spend",
    "category_diversity": "Category diversity", "INCOME_DESC_encoded": "Income band",
    "avg_items_per_basket": "Items per basket", "HOUSEHOLD_SIZE_DESC_encoded": "Household size",
    "AGE_DESC_encoded": "Age band", "exposure_ratio": "Promotion exposure",
    "n_baskets": "Visit frequency", "HH_COMP_DESC_2 Adults Kids": "Household: 2 adults + kids",
    "HH_COMP_DESC_1 Adult Kids": "Household: 1 adult + kids", "HH_COMP_DESC_Single Female": "Household: single female",
    "HH_COMP_DESC_Single Male": "Household: single male", "HH_COMP_DESC_Unknown": "Household: unknown",
    "HOMEOWNER_DESC_Unknown": "Home: unknown", "HOMEOWNER_DESC_Probable Owner": "Home: probable owner",
    "HOMEOWNER_DESC_Probable Renter": "Home: probable renter", "HOMEOWNER_DESC_Renter": "Home: renter",
}

font: DejaVu Sans | transparent: False | save dir: images


## A1. `rank-stability.png` — baseline length N (was `src/image.png`)

In [2]:
rs = read("h4_rank_stability.csv")
ch = read("h4_early_window_choice.csv").iloc[0]
sel = rs[rs["N"] == ch["chosen_early_n"]].iloc[0]

fig, ax = plt.subplots(figsize=(7.4, 4.5))
ax.axvspan(ch["rule_10pct_weeks"], ch["rule_15pct_weeks"], color=tint(VIOLET, 0.12), lw=0)
ax.axhline(0.8, color=INK3, ls=":", lw=1)
ax.plot(rs["N"], rs["spearman_rho"], color=tint(VIOLET, 0.55), lw=2.2, marker="o", ms=6, zorder=3)
for _, r in rs.iterrows():
    if r["N"] != sel["N"]:
        ax.text(r["N"], r["spearman_rho"] + 0.022, f"{r['spearman_rho']:.2f}", ha="center", fontsize=8, color=INK3)
ax.plot(sel["N"], sel["spearman_rho"], "o", ms=12, color=VIOLET, zorder=5)
ax.annotate(f"N = {int(sel['N'])} weeks (kneed elbow)\n\u03c1 = {sel['spearman_rho']:.3f}",
            (sel["N"], sel["spearman_rho"]), xytext=(18, -52), textcoords="offset points",
            fontsize=10, fontweight="bold", color=INK,
            arrowprops=dict(arrowstyle="-", color=INK3, lw=0.8))
ax.text((ch["rule_10pct_weeks"] + ch["rule_15pct_weeks"]) / 2, 0.375,
        f"10\u201315% of the\n{int(ch['total_stable_weeks'])}-week window", ha="center", fontsize=8, color=INK2)
ax.text(rs["N"].min(), 0.8 + 0.008, "\u03c1 = 0.8 reference", ha="left", va="bottom", fontsize=8, color=INK3)
ax.set_ylim(0.35, 0.87)
ax.set_xticks(rs["N"])
ax.set_xlabel("Baseline window length N (weeks)")
ax.set_ylabel("Spearman \u03c1, odd vs even weeks")
head(fig, "Longer baselines rank households more consistently",
     f"Spearman correlation between household ranks from odd and even weeks \u00b7 stable window weeks {int(ch['stable_min_week'])}\u2013{int(ch['stable_max_week'])}")
save(fig, "rank-stability")

saved images/rank-stability.png


## A2. `spend-change-box.png` — change by initial group (was `src/image-4.png`)

In [3]:
st = read("h4_diff_boxplot_stats.csv").set_index("group")
fl = read("h4_diff_boxplot_fliers.csv")
groups = [("top20", "Initial top 20%", ROSE), ("rest80", "Remaining 80%", VIOLET)]

fig, ax = plt.subplots(figsize=(7.0, 4.8))
stats = [dict(med=st.loc[g, "med"], q1=st.loc[g, "q1"], q3=st.loc[g, "q3"], whislo=st.loc[g, "whislo"],
              whishi=st.loc[g, "whishi"], mean=st.loc[g, "mean"], label=lab,
              fliers=fl.loc[fl["group"] == g, "value"].values) for g, lab, _ in groups]
bp = ax.bxp(stats, positions=[1, 2.15], widths=0.42, patch_artist=True, showmeans=True, showfliers=True,
            medianprops=dict(color=INK, lw=2.2), whiskerprops=dict(color=INK3, lw=1.2), capprops=dict(color=INK3, lw=1.2),
            meanprops=dict(marker="D", markerfacecolor="white", markeredgecolor=INK, markersize=6),
            flierprops=dict(marker="o", markersize=3, markerfacecolor=tint(INK3, 0.5), markeredgecolor="none"))
for patch, (_, _, col) in zip(bp["boxes"], groups):
    patch.set_facecolor(tint(col, 0.35)); patch.set_edgecolor(col); patch.set_linewidth(1.6)
ax.axhline(0, color=INK3, ls="--", lw=1)
for x, (g, lab, col) in zip([1, 2.15], groups):
    ax.text(x + 0.30, st.loc[g, "med"], f"median {sgn(st.loc[g, 'med'])}\nmean {sgn(st.loc[g, 'mean'])}\nn = {int(st.loc[g, 'n']):,}",
            va="center", fontsize=9, color=col, fontweight="bold",
            bbox=dict(fc="white", ec="none", alpha=0.92, pad=2.0), zorder=6)
ax.set_xlim(0.55, 3.35)
ax.set_ylabel("Change in household spend, late \u2212 early window")
ax.grid(axis="x", visible=False)
head(fig, "The initial top 20% spent less later; the rest spent more",
     "Per-household change in spend between weeks 17\u201326 and weeks 90\u201399 \u00b7 box = quartiles, diamond = mean")
save(fig, "spend-change-box")

saved images/spend-change-box.png


## A3. `cohort-contribution.png` — growth in the remaining 80% by cohort (was `src/image-6.png`)

In [4]:
cr = read("h4_2_contribution_rest80.csv").set_index("cohort")
cb = read("h4_2_cohort_boxplot_stats.csv").set_index("cohort")
cf = read("h4_2_cohort_boxplot_fliers.csv")
coh = [("existing", "Existing", VIOLET), ("new_or_reactivated", "New or\nreactivated", BLUE)]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.0, 4.6), gridspec_kw=dict(width_ratios=[1, 1.15], wspace=0.28))
vals = [cr.loc[c, "diff_sum"] for c, _, _ in coh]
bars = a1.bar([l for _, l, _ in coh], vals, color=[tint(col, 0.55) for _, _, col in coh],
              edgecolor=[col for _, _, col in coh], linewidth=1.6, width=0.55)
for b, (c, _, col) in zip(bars, coh):
    a1.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{cr.loc[c, 'contribution_pct']:.1f}%\nof the increase",
            ha="center", va="bottom", fontsize=10, fontweight="bold", color=col)
a1.set_ylim(0, max(vals) * 1.22)
a1.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
a1.set_ylabel("Net spend increase, late \u2212 early (sum)")
a1.set_title("Share of the net increase", loc="left", fontsize=10, color=INK2, pad=8)
a1.grid(axis="x", visible=False)

stats = [dict(med=cb.loc[c, "med"], q1=cb.loc[c, "q1"], q3=cb.loc[c, "q3"], whislo=cb.loc[c, "whislo"],
              whishi=cb.loc[c, "whishi"], mean=cb.loc[c, "mean"], label=l,
              fliers=cf.loc[cf["cohort"] == c, "value"].values) for c, l, _ in coh]
bp = a2.bxp(stats, positions=[1, 2.3], widths=0.42, patch_artist=True, showmeans=True,
            medianprops=dict(color=INK, lw=2.2), whiskerprops=dict(color=INK3, lw=1.2), capprops=dict(color=INK3, lw=1.2),
            meanprops=dict(marker="D", markerfacecolor="white", markeredgecolor=INK, markersize=6),
            flierprops=dict(marker="o", markersize=3, markerfacecolor=tint(INK3, 0.5), markeredgecolor="none"))
for patch, (_, _, col) in zip(bp["boxes"], coh):
    patch.set_facecolor(tint(col, 0.35)); patch.set_edgecolor(col); patch.set_linewidth(1.6)
a2.axhline(0, color=INK3, ls="--", lw=1)
for x, (c, _, col) in zip([1, 2.3], coh):
    a2.text(x + 0.28, cb.loc[c, "med"], f"median {sgn(cb.loc[c, 'med'])}", va="center", fontsize=9, color=col, fontweight="bold",
            bbox=dict(fc="white", ec="none", alpha=0.92, pad=2.0), zorder=6)
a2.set_xlim(0.55, 3.5)
a2.set_ylabel("Change per household, late \u2212 early")
a2.set_title("Change per household", loc="left", fontsize=10, color=INK2, pad=8)
a2.grid(axis="x", visible=False)
head(fig, "Existing households supplied most of the growth in the remaining 80%",
     f"Remaining 80% of households (n = {int(cr['count'].sum()):,}) \u00b7 net increase {cr['total_diff_rest'].iloc[0]:,.0f} split by customer status", top=0.78)
save(fig, "cohort-contribution")

saved images/cohort-contribution.png


## A4. `cv-folds.png` — PR-AUC and recall by fold (was `src/image-9.png`)

In [5]:
cvf = read("ml_logit_cv_folds.csv")
cvs = read("ml_logit_cv_summary.csv").set_index("metric")
mets = [("pr_auc", "PR-AUC", VIOLET), ("recall", "Recall", BLUE)]

fig, ax = plt.subplots(figsize=(6.4, 4.6))
for i, (m, lab, col) in enumerate(mets):
    xs = i + (cvf["fold"] - cvf["fold"].mean()) * 0.07
    ax.scatter(xs, cvf[m], s=46, color=tint(col, 0.65), edgecolor=col, linewidth=1, zorder=3)
    mean, sd = cvs.loc[m, "mean"], cvs.loc[m, "std"]
    ax.errorbar(i + 0.30, mean, yerr=sd, fmt="D", color=col, ms=9, capsize=5, lw=1.6, zorder=4)
    ax.text(i + 0.42, mean, f"mean {mean:.3f}\n\u00b1 {sd:.3f} (SD)", va="center", fontsize=9, color=col, fontweight="bold")
ax.set_xticks(range(len(mets))); ax.set_xticklabels([l for _, l, _ in mets], fontsize=11)
ax.set_xlim(-0.55, 1.95)
ax.set_ylim(0.40, 0.82)
ax.set_ylabel("Score (one dot per fold)")
ax.grid(axis="x", visible=False)
head(fig, "Cross-validation results vary by fold",
     f"Logistic regression \u00b7 {len(cvf)}-fold stratified CV \u00b7 499 households")
save(fig, "cv-folds")

saved images/cv-folds.png


## A5. `shap-importance.png` — random forest SHAP importance (was `src/image-14.png`)

In [6]:
sh = read("ml_shap_summary.csv").sort_values("rank", kind="stable")
fig, ax = plt.subplots(figsize=(7.6, 5.6))
y = np.arange(len(sh))[::-1]
top5 = sh["rank"] <= 5
ax.barh(y, sh["mean_abs_shap"], color=[VIOLET if t else tint(VIOLET, 0.3) for t in top5], height=0.66)
for yy, v in zip(y, sh["mean_abs_shap"]):
    ax.text(v + 0.0015, yy, f"{v:.3f}", va="center", fontsize=8, color=INK2)
ax.set_yticks(y); ax.set_yticklabels([LABELS[f] for f in sh["feature"]])
for lab, t in zip(ax.get_yticklabels(), top5):
    lab.set_color(INK if t else INK3); lab.set_fontweight("bold" if t else "normal")
ax.set_xlabel("Mean |SHAP value| (average impact on the predicted risk of dropping out)")
ax.set_xlim(0, sh["mean_abs_shap"].max() * 1.12)
ax.grid(axis="y", visible=False)
head(fig, "What the random forest relies on",
     "SHAP importance, random forest on 499 baseline top-20% households \u00b7 top five in bold", top=0.84)
save(fig, "shap-importance")

saved images/shap-importance.png


## B1. `lorenz-curve.png`

In [7]:
lc = read("h4_lorenz_curve.csv")
sh_ = read("h4_concentration_shares.csv").set_index("top_pct_households")
gi = read("h4_gini.csv").iloc[0]
x = np.r_[0, lc["cum_pct_households"]]; yv = np.r_[0, lc["cum_pct_sales"]]

fig, ax = plt.subplots(figsize=(6.6, 6.4))
ax.fill_between(x, x, yv, color=tint(VIOLET, 0.12), lw=0)
ax.plot([0, 100], [0, 100], ls="--", color=INK3, lw=1.5, label="Perfect equality")
ax.plot(x, yv, color=VIOLET, lw=3.2, label="Observed distribution")
for k in (10, 20, 30):
    v = sh_.loc[k, "cum_share_pct"]
    strong = k == 20
    col = ROSE if strong else INK3
    ax.plot([k, k], [0, v], color=col, lw=1, alpha=0.7); ax.plot([0, k], [v, v], color=col, lw=1, alpha=0.7)
    ax.plot(k, v, "o", color=col, ms=8 if strong else 6, zorder=5)
    ax.text(k + 2.2, v - 4.5, f"top {k}% \u2192 {v:.1f}%", fontsize=10 if strong else 9,
            fontweight="bold" if strong else "normal", color=INK if strong else INK2,
            bbox=dict(fc="white", ec="none", alpha=0.9, pad=1.5), zorder=6)
ax.text(97, 8, f"Gini = {gi['gini']:.3f}", ha="right", fontsize=17, fontweight="bold", color=VIOLET)
ax.legend(loc="upper left", frameon=False, bbox_to_anchor=(0.02, 0.97))
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.set_aspect("equal")
ax.set_xlabel("Cumulative share of households (highest spenders first, %)")
ax.set_ylabel("Cumulative share of revenue (%)")
head(fig, "Spending concentration in the baseline window",
     f"{int(gi['n_households_early_window']):,} households with transactions in weeks 17\u201326", top=0.84)
save(fig, "lorenz-curve")

saved images/lorenz-curve.png


## B2. `odds-ratios.png` — forest plot

In [8]:
od = read("ml_logit_odds_ratios.csv").sort_values("odds_ratio_l2", kind="stable").reset_index(drop=True)
lo, hi, orr = od["or_l2_boot_ci_low"], od["or_l2_boot_ci_high"], od["odds_ratio_l2"]
sig = (hi < 1) | (lo > 1)          # interval excludes 1

fig, ax = plt.subplots(figsize=(8.2, 7.0))
y = np.arange(len(od))[::-1]
ax.axvline(1, color=INK, lw=1.6, zorder=2)
for yy, o, l, h, s in zip(y, orr, lo, hi, sig):
    col = VIOLET if o < 1 else ROSE
    c = col if s else tint(col, 0.35)
    ax.hlines(yy, l, h, color=c, lw=3 if s else 2, zorder=3)
    ax.plot(o, yy, "o", color=c, ms=9 if s else 7, zorder=4)
    ax.text(max(h, 1) * 1.06, yy, f"{o:.2f}", va="center", fontsize=9, fontweight="bold" if s else "normal", color=INK if s else INK3)
ax.set_xscale("log")
ax.set_xticks([0.2, 0.3, 0.5, 0.7, 1, 1.5, 2, 3])
ax.get_xaxis().set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_xlim(0.17, 4.6)
ax.set_yticks(y); ax.set_yticklabels([LABELS[f] for f in od["feature"]])
for lab, s in zip(ax.get_yticklabels(), sig):
    lab.set_color(INK if s else INK3); lab.set_fontweight("bold" if s else "normal")
ax.set_xlabel("Odds ratio of dropping out of the top tier (log scale; 1 = no association)")
ax.grid(axis="y", visible=False)
ax.set_ylim(-1.5, len(od) - 0.4)
ax.text(0.93, -0.95, "◀ protective", ha="right", va="center", fontsize=9, color=VIOLET, fontweight="bold")
ax.text(1.07, -0.95, "riskier ▶", ha="left", va="center", fontsize=9, color=ROSE, fontweight="bold")
head(fig, "What keeps a household in the top spending tier",
     "Logistic regression (L2) on 499 baseline top-20% households \u00b7 dot = odds ratio, line = 95% bootstrap interval\n"
     "Bold = interval excludes 1 \u00b7 numeric features are per 1 SD", top=0.85)
save(fig, "odds-ratios")

saved images/odds-ratios.png


## B3. `shap-saturation.png`

In [9]:
sd_ = read("ml_shap_saturation_deciles.csv")
a = sd_[sd_["feature"] == "sales_early"].reset_index(drop=True)
b = sd_[sd_["feature"] == "category_diversity"].reset_index(drop=True)
# saturation starts at the first decile after which every step is smaller than 5% of the full range
_rng = a["mean_shap"].max() - a["mean_shap"].min()
_small = (a["mean_shap"].diff().abs().shift(-1).fillna(0) < 0.05 * _rng).to_numpy()
_i = next(i for i in range(len(_small)) if _small[i:].all())
flat_from = int(a["decile"].iloc[_i])

fig, ax = plt.subplots(figsize=(7.6, 4.8))
ax.axhline(0, color=INK3, lw=1, ls="--")
ax.axvspan(flat_from - 0.5, a["decile"].max() + 0.5, color=tint(VIOLET, 0.10), lw=0)
ax.set_ylim(a["mean_shap"].min() * 1.25, a["mean_shap"].max() * 1.2)
ax.plot(b["decile"], b["mean_shap"], color=tint(INK3, 0.55), lw=2, marker="o", ms=5, zorder=3)
ax.plot(a["decile"], a["mean_shap"], color=VIOLET, lw=3.2, marker="o", ms=8, zorder=4)
ax.text(b["decile"].iloc[-1] + 0.75, b["mean_shap"].iloc[-1], "Category diversity\n(no flattening\nat the top)",
        fontsize=8.5, color=INK3, ha="left", va="center")
ax.text(a["decile"].iloc[0] + 0.2, a["mean_shap"].iloc[0], "Baseline spend", fontsize=10, fontweight="bold", color=VIOLET, va="center")
d9 = a[a["decile"] == flat_from].iloc[0]; d10 = a[a["decile"] == a["decile"].max()].iloc[0]
ax.text((flat_from + a["decile"].max()) / 2, 0.02, f"flat above {d9['feature_min']:,.0f}\n({sgn(d9['mean_shap'], 3)} \u2192 {sgn(d10['mean_shap'], 3)})",
        ha="center", va="center", fontsize=9, fontweight="bold", color=INK)
ax.set_xticks(a["decile"])
ax.set_xlim(0.5, a["decile"].max() + 2.6)
ax.set_xlabel("Decile of the feature value (1 = lowest)")
ax.set_ylabel("Mean SHAP value (positive = riskier)")
head(fig, "Baseline spend saturates at the top; category diversity keeps mattering",
     f"Random forest on 499 households \u00b7 baseline spend ranges from {a['feature_min'].min():,.0f} to {a['feature_max'].max():,.0f} "
     f"(decile {flat_from} starts at {d9['feature_min']:,.0f})")
save(fig, "shap-saturation")

saved images/shap-saturation.png


## B4. `sensitivity-window.png`

In [10]:
sw = read("h4_sensitivity_window.csv")
_lim = sw["mean"].abs().max() * 1.45
fig, ax = plt.subplots(figsize=(7.0, 4.6))
ax.axhline(0, color=INK3, lw=1, ls="--")
for g, lab, col in [("top", "Initial top 20%", ROSE), ("rest", "Remaining 80%", VIOLET)]:
    s = sw[sw["group_en"] == g].sort_values("early_n", kind="stable")
    ax.plot(s["early_n"], s["mean"], color=col, lw=3, marker="o", ms=8, zorder=3)
    for _, r in s.iterrows():
        ax.text(r["early_n"], r["mean"] + (_lim * 0.05 if r["mean"] > 0 else -_lim * 0.05), sgn(r["mean"]),
                ha="center", va="bottom" if r["mean"] > 0 else "top", fontsize=9,
                fontweight="bold" if r["early_n"] == 10 else "normal", color=col)
    e = s.iloc[-1]
    ax.text(e["early_n"] + 0.18, e["mean"], lab, va="center", fontsize=10, fontweight="bold", color=col)
ax.set_xticks(sorted(sw["early_n"].unique()))
ax.set_xlim(7.6, 13.6)
ax.set_xlabel("Baseline window length (weeks)")
ax.set_ylabel("Mean change in spend, late \u2212 early")
ax.set_ylim(-_lim, _lim)
head(fig, "The reversal holds for 8-, 10- and 12-week baselines",
     "Mean change in household spend by initial group \u00b7 10 weeks is the main analysis (bold)")
save(fig, "sensitivity-window")

saved images/sensitivity-window.png


## C1. `share-bootstrap.png`

In [11]:
sb = read("h4_share_bootstrap.csv").iloc[0]
dr = read("h4_share_bootstrap_draws.csv")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.4, 4.5), gridspec_kw=dict(width_ratios=[1, 1.45], wspace=0.30))
a1.plot([0, 1], [sb["share_early_pct"], sb["share_late_pct"]], color=VIOLET, lw=4, zorder=2)
a1.plot([0, 1], [sb["share_early_pct"], sb["share_late_pct"]], "o", color=VIOLET, ms=13, zorder=3)
a1.text(0, sb["share_early_pct"] + 1.6, f"{sb['share_early_pct']:.2f}%", ha="center", fontsize=12, fontweight="bold")
a1.text(1, sb["share_late_pct"] - 3.2, f"{sb['share_late_pct']:.2f}%", ha="center", fontsize=12, fontweight="bold")
a1.set_xticks([0, 1]); a1.set_xticklabels(["Early window\n(weeks 17\u201326)", "Later window\n(weeks 90\u201399)"])
a1.set_xlim(-0.45, 1.45)
a1.set_ylim(min(sb["share_late_pct"], sb["share_early_pct"]) - 12, max(sb["share_late_pct"], sb["share_early_pct"]) + 8)
a1.set_ylabel("Revenue share of the initial top 20% (%)")
a1.set_title(f"Share fell {abs(sb['share_diff_pp']):.2f} points", loc="left", fontsize=11, fontweight="bold", color=INK, pad=10)
a1.grid(axis="x", visible=False)

a2.hist(dr["share_diff_pp"], bins=50, color=tint(VIOLET, 0.35), edgecolor="white", linewidth=0.6)
a2.axvspan(sb["ci_low_pp"], sb["ci_high_pp"], color=tint(VIOLET, 0.16), lw=0, zorder=0)
a2.axvline(sb["share_diff_pp"], color=INK, lw=2)
a2.axvline(0, color=ROSE, lw=2, ls="--")
top = a2.get_ylim()[1]
a2.set_ylim(0, top * 1.22)
a2.text(sb["share_diff_pp"] + 0.2, top * 1.12, f"{sgn(sb['share_diff_pp'])} pp", ha="left", fontsize=11, fontweight="bold")
a2.text(sb["ci_low_pp"] - 0.15, top * 1.0, neg(f"{sb['ci_low_pp']:.2f}"), ha="right", fontsize=9, color=INK2)
a2.text(sb["ci_high_pp"] + 0.15, top * 1.0, neg(f"{sb['ci_high_pp']:.2f}"), ha="left", fontsize=9, color=INK2)
a2.text(0.15, top * 0.55, "no change", rotation=90, color=ROSE, fontsize=9, va="center")
a2.set_xlim(sb["ci_low_pp"] - 3, 2)
a2.set_xlabel(f"Change in revenue share (percentage points)\n{int(sb['n_boot']):,} bootstrap resamples")
a2.set_ylabel("Resamples")
a2.set_title("The 95% interval excludes zero", loc="left", fontsize=11, fontweight="bold", color=INK, pad=10)
a2.grid(axis="x", visible=False)
head(fig, "The initial top 20% held a smaller share of revenue later",
     f"Share of total spend held by the {int(sb['n_top'])} initial top-20% households among {int(sb['n_panel']):,} with transactions in both windows", top=0.78)
save(fig, "share-bootstrap")

saved images/share-bootstrap.png


## C2. `contribution-split.png`

In [12]:
cr = read("h4_2_contribution_rest80.csv").set_index("cohort")
coh = [("existing", "Existing", VIOLET), ("new_or_reactivated", "New or reactivated", BLUE)]
fig, ax = plt.subplots(figsize=(7.2, 4.6))
w = 0.34
for i, (c, lab, col) in enumerate(coh):
    h = cr.loc[c, "headcount_pct"]; g = cr.loc[c, "contribution_pct"]
    b1 = ax.bar(i - w / 2 - 0.01, h, w, color=tint(col, 0.30), edgecolor=col, linewidth=1.4)
    b2 = ax.bar(i + w / 2 + 0.01, g, w, color=col, edgecolor=col)
    ax.text(i - w / 2 - 0.01, h + 1.2, f"{h:.1f}%", ha="center", fontsize=11, fontweight="bold", color=INK2)
    ax.text(i + w / 2 + 0.01, g + 1.2, f"{g:.1f}%", ha="center", fontsize=11, fontweight="bold", color=col)
ax.set_xticks(range(len(coh))); ax.set_xticklabels([l for _, l, _ in coh], fontsize=11)
ax.set_ylim(0, 85); ax.set_ylabel("Share (%)")
ax.grid(axis="x", visible=False)
ax.legend(handles=[Rectangle((0, 0), 1, 1, fc=tint(INK2, 0.30), ec=INK2, lw=1.4, label="Share of households"),
                   Rectangle((0, 0), 1, 1, fc=INK2, ec=INK2, label="Share of the net spend increase")],
          frameon=False, loc="upper right")
head(fig, "New and reactivated households contributed less than their headcount",
     f"Remaining 80% of households (n = {int(cr['count'].sum()):,}) \u00b7 contribution = share of the net spend increase, late \u2212 early")
save(fig, "contribution-split")

saved images/contribution-split.png


## C3. `pr-curve.png`

In [13]:
pr = read("ml_pr_curve.csv").sort_values("threshold", ascending=False, kind="stable")
bl = read("ml_baselines.csv").iloc[0]
cm = read("ml_logit_confusion.csv").iloc[0]

fig, ax = plt.subplots(figsize=(6.6, 5.2))
ax.plot(pr["recall"], pr["precision"], color=VIOLET, lw=1.8, label="Logistic regression (out-of-fold predictions)")
ax.axhline(bl["dummy_pr_auc"], color=INK3, ls="--", lw=1.6, label=f"Stratified dummy classifier (PR-AUC {bl['dummy_pr_auc']:.3f})")
ax.plot(cm["recall"], cm["precision"], "o", color=ROSE, ms=10, zorder=5)
ax.annotate(f"threshold {cm['threshold']:.1f}\nrecall {cm['recall']:.3f}, precision {cm['precision']:.3f}",
            (cm["recall"], cm["precision"]), xytext=(14, 26), textcoords="offset points", fontsize=9, color=ROSE,
            fontweight="bold", arrowprops=dict(arrowstyle="-", color=ROSE, lw=0.8))
ax.text(0.04, 0.37, f"PR-AUC {bl['logit_pr_auc']:.3f} (5-fold mean)", fontsize=10, fontweight="bold", color=VIOLET)
ax.set_xlim(0, 1.02); ax.set_ylim(0.3, 1.02)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.legend(frameon=False, loc="upper right", fontsize=8.5)
head(fig, "Precision\u2013recall curve, logistic regression",
     "Predicting which baseline top-20% households drop out of the top tier \u00b7 499 households", top=0.84)
save(fig, "pr-curve")

saved images/pr-curve.png


## C4. `confusion-matrix.png`

In [14]:
cm = read("ml_logit_confusion.csv").iloc[0]
M = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])
fig, ax = plt.subplots(figsize=(6.0, 5.0))
fills = [[tint(VIOLET, 0.22), tint(INK3, 0.18)], [ROSE, tint(VIOLET, 0.55)]]
for i in range(2):
    for j in range(2):
        ax.add_patch(Rectangle((j, 1 - i), 0.96, 0.96, fc=fills[i][j], ec="none"))
        txtcol = "white" if (i, j) == (1, 0) else INK
        ax.text(j + 0.48, 1 - i + 0.56, f"{int(M[i, j])}", ha="center", va="center", fontsize=28, fontweight="bold", color=txtcol)
names = [["True negative", "False positive"], ["False negative (missed)", "True positive"]]
for i in range(2):
    for j in range(2):
        ax.text(j + 0.48, 1 - i + 0.22, names[i][j], ha="center", va="center", fontsize=9,
                color="white" if (i, j) == (1, 0) else INK2)
ax.set_xlim(0, 2); ax.set_ylim(0, 2.0); ax.set_aspect("equal"); ax.axis("off")
for j, t in enumerate(["Predicted: retained", "Predicted: dropped out"]):
    ax.text(j + 0.48, 2.05, t, ha="center", va="bottom", fontsize=10, color=INK2)
for i, t in enumerate(["Actual:\nretained", "Actual:\ndropped out"]):
    ax.text(-0.04, 1 - i + 0.48, t, ha="right", va="center", fontsize=10, color=INK2)
head(fig, f"{int(cm['fn'])} of {int(cm['n_actual_shrink'])} households that dropped out were missed",
     f"Logistic regression, out-of-fold predictions at threshold {cm['threshold']:.1f} \u00b7 false-negative rate {cm['fn_rate_pct']:.1f}% \u00b7 499 households", top=0.78)
save(fig, "confusion-matrix")

saved images/confusion-matrix.png
